In [ ]:
import os
import glob
import json
from fastai.vision.all import *

In [2]:
segmentation_dir = "../data/output/segmentation/"
model_path = "../data/models/resnet50_fish.pkl" 
output_nlp_dir = "../data/output/nlp_payload"

os.makedirs(output_nlp_dir, exist_ok=True)

In [3]:
learn = load_learner(model_path)

/Users/filimono/Documents/Inno/Interactive-fish-study-system/.venv/lib/python3.13/site-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


AttributeError: Custom classes or functions exported with your `Learner` not available in namespace. Re-declare/import before loading:
	'Resolver' object has no attribute '__dict__'

In [ ]:
search_pattern = os.path.join(segmentation_dir, "*.jpg")
segmented_images = glob.glob(search_pattern)

In [ ]:
if not segmented_images:
    print(f"В папке {segmentation_dir} не найдено ни одного изображения для классификации.")
else:
    print(f"Найдено изображений от SAM 2: {len(segmented_images)}\n")
    
    
    for img_path in segmented_images:
        image_name = os.path.splitext(os.path.basename(img_path))[0]
        output_json_path = os.path.join(output_nlp_dir, f"{image_name}_nlp.json")
        
        pred_class, pred_idx, probs = learn.predict(img_path)
        confidence = float(probs[pred_idx])
        
        nlp_data = {
            "detected_objects": [
                {
                    "class": pred_class,
                    "confidence": round(confidence, 2),
                    "source": "SAM 2 + ResNet-50", 
                    "image_path": img_path
                }
            ]
        }
        
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(nlp_data, f, indent=4, ensure_ascii=False)
            
        print(f"Готово! JSON сохранен: {output_json_path}\n")

print("Все изображения из папки сегментации успешно классифицированы!")